# RankLab: KuaiRand-Pure baseline experiment on Kaggle

Enable a **GPU accelerator** in Kaggle settings. Add a Kaggle dataset containing the official `KuaiRand-Pure/data/` directory, or set `DOWNLOAD_IF_MISSING = True` and enable Internet. Optionally attach the private `ranklab-baseline-artifacts` dataset as an input: it restores completed derived artifacts and runs only missing or incompatible stages. Raw data is never packaged in the artifact cache.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/kushc2004/rank-lab.git'
WORKDIR = Path('/kaggle/working/rank-lab')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print(WORKDIR)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running BPR-MF.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Preferred: attach a Kaggle dataset whose root contains KuaiRand-Pure/data.
# Optional: download only from the official Zenodo source when Kaggle Internet is enabled.
DOWNLOAD_IF_MISSING = False
expected = ('log_random_4_22_to_5_08_pure.csv', 'log_standard_4_08_to_4_21_pure.csv', 'log_standard_4_22_to_5_08_pure.csv')
candidates = list(Path('/kaggle/input').glob('*/KuaiRand-Pure/data'))
if candidates:
    RAW_DIR = candidates[0]
elif DOWNLOAD_IF_MISSING:
    subprocess.run(['bash', 'scripts/download_kuairand_pure.sh'], check=True)
    RAW_DIR = WORKDIR / 'data/raw/KuaiRand-Pure/data'
else:
    raise FileNotFoundError('Attach the official KuaiRand-Pure Kaggle dataset, or enable Internet and set DOWNLOAD_IF_MISSING=True.')
missing = [name for name in expected if not (RAW_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Not an official KuaiRand-Pure data directory: missing {missing}')
print('Using raw data:', RAW_DIR)

In [ ]:
# Optional cache restore. Attach the RankLab artifact dataset as an input to reuse
# manifests, historical features, models, predictions, metrics, and reports.
artifact_archives = list(Path('/kaggle/input').glob('*/ranklab_artifacts.tar.gz'))
if artifact_archives:
    subprocess.run(['python', 'scripts/restore_kaggle_artifacts.py', str(artifact_archives[0])], check=True)
else:
    print('No artifact cache attached; stages will run as needed.')

In [ ]:
# The runner invokes stages one at a time and skips only compatible, complete artifacts.
command = ['python', 'scripts/run_cached_baselines.py', f'raw_dir={RAW_DIR}', 'device=cuda']
print('$', ' '.join(map(str, command)))
subprocess.run(command, check=True)
subprocess.run(['pytest'], check=True)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((WORKDIR / 'outputs/reports/initial_exposure_gap.md').read_text()))

In [ ]:
# Always leave a compact derived-artifact archive in the notebook output.
# To update the private cache dataset directly, add Kaggle API credentials as secrets
# and set PUBLISH_CACHE=True. Otherwise save this notebook version and attach its
# output archive as an input on the next run.
PUBLISH_CACHE = False
publish = ['python', 'scripts/publish_kaggle_artifacts.py']
if not PUBLISH_CACHE:
    publish.append('--no-upload')
subprocess.run(publish, check=True)
print(WORKDIR / 'artifacts/kaggle/ranklab_artifacts.tar.gz')